In [ ]:
import numpy as np
from freshflow.loader import load_sales
from freshflow.validation import column
from freshflow.metrics import build_metrics, summarize

clean = load_sales("../data/processed/freshflow_clean.csv", expected_rows=None)
metrics = build_metrics(clean)
print(clean.shape)

In [ ]:
for name, values in metrics.items():
    print(f"{name:20}", np.round(values[:5], 2))

In [ ]:
summary = summarize(clean, metrics)
for name, value in summary.items():
    print(f"{name:28} {value:,.2f}")

In [ ]:
quantity = column(clean, "quantity")
price = column(clean, "unit_price")
cost = column(clean, "unit_cost")
rate = column(clean, "discount_rate")

for i in range(5):
    gross = quantity[i] * price[i]
    net = gross - gross * rate[i]
    profit = net - quantity[i] * cost[i]
    print(i, round(gross, 2), round(net, 2), round(profit, 2))
    np.testing.assert_allclose(metrics["net_revenue"][i], net)
    np.testing.assert_allclose(metrics["gross_profit"][i], profit)

In [ ]:
missing_price = np.isnan(price)
print(metrics["net_revenue"][missing_price])
print(metrics["gross_profit"][missing_price])

In [ ]:
print("Overall margin (correct):", round(summary["gross margin %"], 2))
print("Average of row margins (wrong):", round(np.nanmean(metrics["margin_pct"]), 2))

In [ ]:
from freshflow.stats import (
    describe, iqr_fences, histogram_table, group_performance, ranking,
    OUTLET_NAMES, PRODUCT_NAMES, CATEGORY_NAMES,
)

selected = {
    "net_revenue": metrics["net_revenue"],
    "gross_profit": metrics["gross_profit"],
    "quantity": column(clean, "quantity"),
}

for name, values in selected.items():
    print(name)
    for key, value in describe(values).items():
        print(f"  {key:14} {value:12,.2f}")

In [ ]:
revenue = metrics["net_revenue"][~np.isnan(metrics["net_revenue"])]

print("Mean:", round(revenue.mean(), 2))
print("Median:", round(np.median(revenue), 2))
print("Std (population, ddof=0):", round(revenue.std(ddof=0), 2))
print("Std (sample, ddof=1):", round(revenue.std(ddof=1), 2))
print("Percentiles 90/95/99:", np.round(np.percentile(revenue, [90, 95, 99]), 2))
print("Quantile 0.5:", round(np.quantile(revenue, 0.5), 2))

In [ ]:
low, high = iqr_fences(metrics["net_revenue"])
outlier_mask = (metrics["net_revenue"] < low) | (metrics["net_revenue"] > high)
print("Fences:", round(low, 2), round(high, 2))
print("Statistical outliers:", outlier_mask.sum())

top_rows = np.argsort(np.nan_to_num(metrics["net_revenue"], nan=-1))[::-1][:5]
print(clean[top_rows][:, [0, 2, 3, 5, 6, 8]])

In [ ]:
counts, edges = histogram_table(metrics["net_revenue"], bins=8)
for count, left, right in zip(counts, edges[:-1], edges[1:]):
    print(f"{left:8.1f} - {right:8.1f} : {count}")

In [ ]:
def show(result, names):
    for i, key in enumerate(result["keys"]):
        print(f"{names[key]:22} units={result['units'][i]:8,.0f}  revenue={result['revenue'][i]:12,.2f}  "
              f"profit={result['profit'][i]:12,.2f}  margin={result['margin_pct'][i]:6.2f}%  "
              f"share={result['revenue_share_pct'][i]:6.2f}%")

outlet = group_performance(clean, metrics, "outlet_id")
product = group_performance(clean, metrics, "product_code")
category = group_performance(clean, metrics, "category_code")

show(outlet, OUTLET_NAMES)
print()
show(product, PRODUCT_NAMES)
print()
show(category, CATEGORY_NAMES)

In [ ]:
for field in ["revenue", "profit", "units"]:
    print(field, [OUTLET_NAMES[k] for k in ranking(outlet, field)])

for field in ["revenue", "profit", "units"]:
    print(field, [PRODUCT_NAMES[k] for k in ranking(product, field)])

In [ ]:
product_codes = column(clean, "product_code")
for key in product["keys"]:
    mask = product_codes == key
    print(f"{PRODUCT_NAMES[key]:22} median price={np.nanmedian(price[mask]):7.2f}  units={product['units'][list(product['keys']).index(key)]:6,.0f}")